In [1]:
#EEL
import os
import gradio as gr
from PIL import Image
import glob
import tkinter as tk
from tkinter import filedialog
import threading

def select_folder(folder_type):
    """Open a folder selection dialog and return the selected path"""
    # Run tkinter in a thread to prevent blocking Gradio
    result = [None]  # Use list to store result from thread
    
    def select():
        root = tk.Tk()
        root.withdraw()
        if folder_type == "input":
            title = "Select Input Folder"
        else:
            title = "Select Output Folder"
        folder = filedialog.askdirectory(title=title)
        result[0] = folder
        root.destroy()
    
    # Run folder dialog in separate thread
    thread = threading.Thread(target=select)
    thread.start()
    thread.join()
    
    return result[0]

def process_images(input_folder, output_folder):
    """Process all images in input_folder and save to output_folder"""
    if not input_folder or not os.path.exists(input_folder):
        return "Please select a valid input folder."
    
    if not output_folder:
        return "Please select a valid output folder."
    
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        try:
            os.makedirs(output_folder)
        except Exception as e:
            return f"Error creating output folder: {e}"
    
    # Target dimensions
    target_width, target_height = 446, 850
    
    # Get all image files in the input folder
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.gif', '*.tiff']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(input_folder, ext)))
    
    if not image_files:
        return f"No image files found in {input_folder}"
    
    processed_count = 0
    error_count = 0
    
    # Process each image
    for img_path in image_files:
        try:
            # Open the image
            img = Image.open(img_path)
            
            # Get original dimensions
            orig_width, orig_height = img.size
            
            # Calculate aspect ratios
            target_ratio = target_width / target_height
            orig_ratio = orig_width / orig_height
            
            # Determine if we need to crop
            if orig_ratio > target_ratio:
                # Image is wider than target ratio, crop width
                new_width = int(orig_height * target_ratio)
                left = (orig_width - new_width) // 2
                right = left + new_width
                img = img.crop((left, 0, right, orig_height))
            elif orig_ratio < target_ratio:
                # Image is taller than target ratio, crop height
                new_height = int(orig_width / target_ratio)
                top = (orig_height - new_height) // 2
                bottom = top + new_height
                img = img.crop((0, top, orig_width, bottom))
            
            # Resize to target dimensions
            img = img.resize((target_width, target_height), Image.LANCZOS)
            
            # Save the processed image
            filename = os.path.basename(img_path)
            output_path = os.path.join(output_folder, filename)
            img.save(output_path)
            
            processed_count += 1
            
        except Exception as e:
            error_count += 1
            print(f"Error processing {img_path}: {e}")
    
    result = f"Processed {processed_count} of {len(image_files)} images.\n"
    if error_count > 0:
        result += f"{error_count} images failed to process.\n"
    result += f"Saved to: {output_folder}"
    
    return result

def select_input():
    return select_folder("input")

def select_output():
    return select_folder("output")

# Create Gradio interface
with gr.Blocks(title="Image Resizer") as app:
    gr.Markdown("# Image Resizer (446 x 850px)")
    gr.Markdown("This tool resizes all images in a folder to 446 x 850 pixels, keeping objects centered.")
    
    with gr.Row():
        input_folder = gr.Textbox(label="Input Folder Path")
        input_btn = gr.Button("Browse")
    
    with gr.Row():
        output_folder = gr.Textbox(label="Output Folder Path")
        output_btn = gr.Button("Browse")
    
    process_btn = gr.Button("Process Images", variant="primary")
    result = gr.Textbox(label="Results", lines=5)
    
    # Connect buttons to functions
    input_btn.click(fn=select_input, outputs=input_folder)
    output_btn.click(fn=select_output, outputs=output_folder)
    process_btn.click(fn=process_images, inputs=[input_folder, output_folder], outputs=result)

if __name__ == "__main__":
    app.launch()

c:\Users\r.musawi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
